# Exposure time calculator: SN


***

## 0. Your spectrum is a SN Ia

If case of SN Ia, there is a template (e.g., salt or twin) able to generate spectra given inoput parameters, such as `c` and `x1` (salt).

The spectrum generation has been implemented as part of `point_source` of `scene`. It is then easy to interact with it within lazuli.

Let's imagine we want to trigger `z=1.2`, `x1=1.5`, `c=0.2` salt-modelled SN Ia at `phase=-2.2`

## 1. Load a SN Ia

Let's create a `LazuliSNIa` with given properties

In [1]:
import numpy as np
import slicersim

In [2]:
target = slicersim.LazuliSN(model="salt", redshift=1.2, x1=1.5, c=0.2, phase=-2.2)

In [3]:
# let's check if recorded
target.get_properties(["redshift", "c", "x1"])

{'redshift': 1.2, 'c': 0.2, 'x1': 1.5}

## 2. `setup_` the system

Let's say you want to reach an average signal-to-noise ratio of 20 within the [4000, 7000] angstrom wavelength range. 

In [4]:
# set inplace=False to avoid setting target to the returned configuration.
config, reached_snr = target.setup_to_snr(20, lbda_range=[4000, 7000], frame='rest', 
                                          statistic=np.nanmean)
# this is the best config
print(config)

{'nmd': (64, 8, 0), 'nramp': 7}


## 3. `get_` what you want.

In [5]:
target.get_exposure_time()

10142.720000000001

In [6]:
target.get_exposure_time(full=True)

{'integration_time': 1426.32,
 'exposure_time': 1448.96,
 'tframe': 2.83,
 'tgroup': 22.64,
 'total_exptime': 10142.720000000001}

## 4. `set_` properties of your target or `change_` your instrument

let's imagine that you want to 
- set the magnitude of the target to `mag=21` in `sdssr` band.
- change the read-out mode of your detector to have 4 frames per group with at most 80 groups per ramp

In [7]:
target.set_properties(redshift=0.8, c=0.1, x1=-1.4, phase=0.5)

In [8]:
# here let's set 1 group, no drop, 4 frames per group. 
# The 1 group will be updated with setup_to_snr
target.change_detector_mode(nmd=(1,4,0), max_group=80)

In [9]:
config, reached_snr = target.setup_to_snr(20)
config

{'nmd': (80, 4, 0), 'nramp': 5}

In [10]:
target.get_exposure_time()

4528.0

***
## top-level `slicersim.lazuli_sn_etc()`

The top-level function `slicersim.lazuli_sn_etc()` runs all that for you at once.

In [14]:
exptime, snia_target = slicersim.lazuli_sn_etc(snr=20, model="salt", redshift=1.2, x1=1.5, c=0.2, phase=-2.2)

In [15]:
exptime

10142.720000000001

In [17]:
lbda, flux, variance = snia_target.get_spectrum()